# NLP 主題分群分析（v7.2 — 純非監督版）

## 修正說明

本版在 v6.2 基礎上進行兩項重大改動：

| # | 問題 | 修正方式 |
|---|------|----------|
| F | Cell 13 使用人工標籤做監督評估，違背非監督學習原則 | **完全移除人工標籤流程**；改以 KMeans 群內相對 TF-IDF 自動提取各群特徵詞，並加入 Bootstrap ARI 分群穩定性評估 |
| G | BERTopic 各主題關鍵字大量重複（同一詞出現在多個主題） | **雙層去重**：① BERTopic 內建 `MaximalMarginalRelevance`（主題內語意去重，`diversity=0.4`）；② 展示層跨主題公共詞排除（出現在 ≥ 2 個主題的詞從顯示清單移除，改補主題特有詞） |
| H | 分群品質僅用 Silhouette 單一指標 | 新增 **Calinski-Harabasz** 與 **Davies-Bouldin** 指標，提供三角驗證 |

> 其餘 v6.2 的所有修正（多語言 embedding、jieba 斷詞、停用詞擴充、欄位偵測、n_init=10…）均完整保留。


## Cell 1｜安裝相依套件

In [ ]:
# 安裝相依套件（Colab 環境；本機已安裝可略過）
# [修正 BUG-1] 加入 scikit-learn>=1.0，確保 CountVectorizer(token_pattern=None) 可用
import subprocess, sys
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'bertopic', 'jieba', 'sentence-transformers', 'umap-learn',
     'scikit-learn>=1.0'],   # token_pattern=None 需要 sklearn >= 1.0
    check=False
)
print("✓ 套件安裝完成")


✓ 套件安裝完成


## Cell 2｜匯入套件與全域設定

In [ ]:
# 統一使用 import re，移除 _re 別名
import re
import jieba
import numpy as np
import pandas as pd
from collections import Counter, defaultdict
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    adjusted_rand_score,
    normalized_mutual_info_score,
)
from sklearn.feature_extraction.text import CountVectorizer
from bertopic import BERTopic
from bertopic.representation import MaximalMarginalRelevance
from sentence_transformers import SentenceTransformer

# 固定全域隨機種子，確保結果可重現
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
N_CLUSTERS   = 5

print("✓ 所有套件匯入完成")
print(f"  jieba  版本: {jieba.__version__}")
import sklearn; print(f"  sklearn 版本: {sklearn.__version__}")


/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:46: SyntaxWarning: invalid escape sequence '\s'
  re_skip_default = re.compile("(\r\n|\s)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/finalseg/__init__.py:78: SyntaxWarning: invalid escape sequence '\.'
  re_skip = re.compile("([a-zA-Z0-9]+(?:\.\d+)?%?)")


✓ 所有套件匯入完成
  jieba  版本: 0.42.1
  sklearn 版本: 1.6.1


## Cell 3｜載入原始資料（data4）

In [ ]:
DATA4_URL = (
    'https://raw.githubusercontent.com/411351242/'
    'my-little-school-nlp-project/refs/heads/main/'
    'combined_threads_posts.csv'
)
data4 = pd.read_csv(DATA4_URL)
print('data4 資料筆數:', len(data4))
print('data4 欄位:', list(data4.columns))
data4.head()


data4 資料筆數: 715
data4 欄位: ['貼文編號', '文字內容']


,貼文編號,文字內容
0,貼文1,為什麼固收交易員比經濟學家更快察覺市場變化？\n我之前還在投行的時候，\n碰上過一位頂著數學...
1,貼文2,一些雜談，好久沒寫點正經的東西了XD\n各位讀者將就將就看唄，也請各方先進不吝賜教\n（利率...
2,貼文3,最近這一個月啊，我觀察到一個滿有趣的現象，不知道大家有沒有同感\n平日下午去星巴克買杯咖啡，...
3,貼文4,記得十幾年前，我們在看像Apple這種 to C的公司時，投資邏輯相對單純滿多的～\n你要研...
4,貼文5,每次看到網路上或是身邊有人在那邊到處公開自己的財務狀況，或是狂曬獲利的對帳單，我心裡大概會覺...


## Cell 4｜欄位偵測 + 資料清理（data5）

> **[修正 B]** 原版先套用 `clean_text`，再做欄位自動偵測。
> 若原始欄位名不是「文字內容」，`data5['文字內容']` 這行就會 `KeyError`。
> 本版先偵測/重命名欄位，再套用 `clean_text`，順序正確。


In [ ]:
# ── 步驟 1：自動偵測文字欄位 ──────────────────────────────────
data = data4.copy()
print('原始欄位列表:', list(data.columns))

TEXT_COL_CANDIDATES = ['文字內容', '內容', 'content', 'text', 'body', 'post']
text_col = next((c for c in TEXT_COL_CANDIDATES if c in data.columns), None)
if text_col is None:
    raise ValueError(
        f'找不到文字欄位，目前欄位為：{list(data.columns)}\n'
        f'請手動設定：text_col = "你的欄位名"'
    )
if text_col != '文字內容':
    data = data.rename(columns={text_col: '文字內容'})
    print(f'✓ 文字欄位「{text_col}」→「文字內容」')
else:
    print('✓ 文字欄位「文字內容」存在')

# ── 步驟 2：自動偵測貼文編號欄位 ─────────────────────────────
ID_COL_CANDIDATES = ['貼文編號', 'id', 'ID', 'post_id', '編號', 'thread_id']
id_col = next((c for c in ID_COL_CANDIDATES if c in data.columns), None)
if id_col is None:
    print('⚠ 找不到貼文編號欄位，自動產生流水號「貼文0, 貼文1...」')
    data['貼文編號'] = ['貼文' + str(i) for i in range(len(data))]
elif id_col != '貼文編號':
    data = data.rename(columns={id_col: '貼文編號'})
    print(f'✓ 貼文編號欄位「{id_col}」→「貼文編號」')
else:
    print('✓ 貼文編號欄位「貼文編號」存在')

# ── 步驟 3：定義並套用 clean_text ─────────────────────────────
# [修正 A] clean_text 統一使用 re（不再用 _re 別名）
def clean_text(text):
    """移除換行符號與連載標記 (續)/(续)。"""
    if not isinstance(text, str):
        return text
    text = text.replace('\r\n', '').replace('\r', '').replace('\n', '')
    text = re.sub(r'[（(]\s*[續续]\s*[)）]', '', text)
    text = re.sub(r'  +', ' ', text).strip()
    return text

data5 = data.copy()
data5['文字內容'] = data5['文字內容'].apply(clean_text)

# ── 步驟 4：移除空白列 ────────────────────────────────────────
before = len(data5)
data5 = data5.dropna(subset=['文字內容'])
data5 = data5[data5['文字內容'].str.strip().ne('')].reset_index(drop=True)
print(f'\n有效資料筆數: {len(data5)}（移除 {before - len(data5)} 筆 NaN 或空白文字）')

# ── 驗證清理效果 ──────────────────────────────────────────────
print('\n--- 清理前（第 0 筆，前 150 字）---')
print(repr(data4.iloc[0, data4.columns.get_loc(text_col)][:150]))
print('\n--- 清理後（第 0 筆，前 150 字）---')
print(repr(data5['文字內容'].iloc[0][:150]))
data5.head()


原始欄位列表: ['貼文編號', '文字內容']
✓ 文字欄位「文字內容」存在
✓ 貼文編號欄位「貼文編號」存在

有效資料筆數: 715（移除 0 筆 NaN 或空白文字）

--- 清理前（第 0 筆，前 150 字）---
'為什麼固收交易員比經濟學家更快察覺市場變化？\n我之前還在投行的時候，\n碰上過一位頂著數學 PhD 光環的主管\n他對數字的敏感度驚人、反應又快￼\n套一句他講過的話：「市場的 Credit Spread 變化，\n就像地震前的前兆，不是每個人都聽得見。」\n這位前輩很快得到了老闆的青睞，\n因為他建模的數學模'

--- 清理後（第 0 筆，前 150 字）---
'為什麼固收交易員比經濟學家更快察覺市場變化？我之前還在投行的時候，碰上過一位頂著數學 PhD 光環的主管他對數字的敏感度驚人、反應又快￼套一句他講過的話：「市場的 Credit Spread 變化，就像地震前的前兆，不是每個人都聽得見。」這位前輩很快得到了老闆的青睞，因為他建模的數學模型對固定收益市'


,貼文編號,文字內容
0,貼文1,為什麼固收交易員比經濟學家更快察覺市場變化？我之前還在投行的時候，碰上過一位頂著數學 PhD...
1,貼文2,一些雜談，好久沒寫點正經的東西了XD各位讀者將就將就看唄，也請各方先進不吝賜教（利率曲線的一...
2,貼文3,最近這一個月啊，我觀察到一個滿有趣的現象，不知道大家有沒有同感平日下午去星巴克買杯咖啡，結果...
3,貼文4,記得十幾年前，我們在看像Apple這種 to C的公司時，投資邏輯相對單純滿多的～你要研究這...
4,貼文5,每次看到網路上或是身邊有人在那邊到處公開自己的財務狀況，或是狂曬獲利的對帳單，我心裡大概會覺...


## Cell 5｜切分訓練 / 測試集

In [ ]:
wt_train, wt_test = train_test_split(
    data5, test_size=0.1, random_state=RANDOM_STATE
)
print(f'訓練集: {len(wt_train)} 筆，測試集: {len(wt_test)} 筆')


訓練集: 643 筆，測試集: 72 筆


## Cell 6｜文字向量化（Sentence-Transformers）

使用 `paraphrase-multilingual-MiniLM-L12-v2`（原生支援中文 + 50 種語言）。
原版使用 `all-MiniLM-L6-v2`（英文 only），中文向量品質差，v6 已修正。


In [ ]:
embedder = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

train_docs = wt_train['文字內容'].astype(str).tolist()
test_docs  = wt_test['文字內容'].astype(str).tolist()

train_emb = embedder.encode(train_docs, show_progress_bar=True)
test_emb  = embedder.encode(test_docs,  show_progress_bar=True)

print('訓練集向量維度:', train_emb.shape)
print('測試集向量維度:', test_emb.shape)


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/21 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

訓練集向量維度: (643, 384)
測試集向量維度: (72, 384)


## Cell 7｜KMeans 分群

> **[修正 C]** 原版 `n_init='auto'` 在 sklearn < 1.2 不合法，改為 `n_init=10`。


In [ ]:
# [修正 C] n_init='auto' → n_init=10，向下相容 sklearn < 1.2
km = KMeans(
    n_clusters=N_CLUSTERS,
    n_init=10,           # 原本 'auto' 在舊版 sklearn 會報錯
    init='k-means++',
    random_state=RANDOM_STATE,
)
km.fit(train_emb)

# 加 .copy() 防止 SettingWithCopyWarning
wt_train = wt_train.copy()
wt_test  = wt_test.copy()
wt_train['cluster'] = km.labels_
wt_test['cluster']  = km.predict(test_emb)

print('訓練集各群數量:')
print(wt_train['cluster'].value_counts().sort_index())


訓練集各群數量:
cluster
0    122
1    150
2    169
3    133
4     69
Name: count, dtype: int64


## Cell 8｜分群品質評估（三項非監督指標）

資料無人工標籤，使用**三項內部指標**評估分群品質，互相交叉驗證：

| 指標 | 越好 | 概念 |
|---|---|---|
| **Silhouette Score** | 越高（上限 1） | 群內緊密度 vs 群間距離 |
| **Calinski-Harabasz** | 越高 | 群間離散度 / 群內離散度之比 |
| **Davies-Bouldin** | 越低（下限 0） | 最差鄰近群對的相似度 |

同時列出 k=2~10 的三指標對照表，協助選擇最佳 k。


In [ ]:
# ── 三項內部指標（全部為非監督，不需任何標籤）──────────────────────────────
sil = silhouette_score(train_emb, km.labels_)
ch  = calinski_harabasz_score(train_emb, km.labels_)
db  = davies_bouldin_score(train_emb, km.labels_)

print(f'KMeans (k={N_CLUSTERS}) 內部指標:')
print(f'  Silhouette Score      : {sil:.4f}  ← [-1,1]，越高越好')
print(f'  Calinski-Harabasz     : {ch:.2f}  ← 越高越好（群內緊、群間遠）')
print(f'  Davies-Bouldin        : {db:.4f}  ← 越低越好（最優為 0）')

print('\n不同 k 的三指標對照（協助選擇最佳 k）：')
print(f'  {"k":>3}  {"Silhouette":>12}  {"CH Score":>12}  {"DB Score":>10}')
print(f'  {"-"*3}  {"-"*12}  {"-"*12}  {"-"*10}')
for k in range(2, 11):
    km_k = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(train_emb)
    s = silhouette_score(train_emb, km_k.labels_)
    c = calinski_harabasz_score(train_emb, km_k.labels_)
    d = davies_bouldin_score(train_emb, km_k.labels_)
    marker = ' ← 目前設定' if k == N_CLUSTERS else ''
    print(f'  k={k:2d}  {s:12.4f}  {c:12.2f}  {d:10.4f}{marker}')


KMeans (k=5) 內部指標:
  Silhouette Score      : 0.0367  ← [-1,1]，越高越好
  Calinski-Harabasz     : 26.30  ← 越高越好（群內緊、群間遠）
  Davies-Bouldin        : 3.9476  ← 越低越好（最優為 0）

不同 k 的三指標對照（協助選擇最佳 k）：
    k    Silhouette      CH Score    DB Score
  ---  ------------  ------------  ----------
  k= 2        0.0750         55.51      3.3524
  k= 3        0.0690         36.69      3.4690
  k= 4        0.0412         30.44      4.0985
  k= 5        0.0367         26.30      3.9476 ← 目前設定
  k= 6        0.0341         23.09      3.8477
  k= 7        0.0336         20.87      3.7312
  k= 8        0.0359         19.24      3.6964
  k= 9        0.0301         17.50      3.6190
  k=10        0.0294         16.57      3.6389


## Cell 9｜中文斷詞設定 + BERTopic 主題建模

### 停用詞與 jieba 設定
此格定義 `STOPWORDS` 與 `jieba_tokenizer`，**重新修改停用詞後必須重新執行此格**，
再執行下方 BERTopic 訓練格，否則關鍵字不會更新。


In [ ]:
# 擴充版停用詞表（全部轉 lower，確保大小寫一致）
STOPWORDS = set(w.lower() for w in """
的 了 是 我 你 他 她 它 們 在 也 和 與 就 都 而 及 或
這 那 有 沒 不 很 把 被 讓 從 對 為 以 之 其 並 但 還 又 會 能 要 個
一 二 三 上 下 中 後 前 時 說 講 想 看 做 用 啊 喔 嗎 呢 吧 啦 喇
我們 你們 他們 一個 這個 那個 因為 所以 如果 但是 而且 然後 可能 已經
什麼 怎麼 這樣 那樣 自己 一些 知道 覺得 真的 其實 就是
不是 只是 而是 還是 不會 不能 沒有 有些 這些 那些 這種 那種 有人
時候 很多 幾個 一點 出來 看到 開始 根本 甚至 真正 東西 地方 一樣
他的 我的 你的 大家 各位 之後 之前 目前 現在 一直 應該 必須
比較 非常 越來越 越 更 最 太 又 再 才 卻 仍 依然 反而 例如
針對 關於 透過 經過 包含 包括 以及 等等 之類 方面 部分 整個 所有
再來 預期 回答 問題 來說 來看 來講 一下 而言 總之 接下來 此外 另外
首先 其次 最後 第一 第二 第三 換句話說 也就是說 換言之 簡單來說 進一步
當然 當然啦 所以啊 那沒關係 但你要 我只能說 人性就是這樣 為什麼
回答這個問題之前 回答問題之前 講這個問題 講一下 先講 先說 補充一下 補充說明
這邊補充 整體來說 整體而言 大概就是 大概是 就這樣 就醬 OK啦 好了
anyway anyhow basically btw lol omg wtf imo tbh fwiw tldr
""".split())

# [修正 D3] 擴充英文縮寫黑名單，加入財經常見英文縮寫
BAD_EN = {
    'ex', 're', 'vs', 'etc', 'aka', 'ie', 'eg',
    'tf', 'gdp', 'ai', 'us', 'uk', 'eu', 'un',
    'ipo', 'etf', 'esg', 'roi', 'cpi', 'ppi',
    'yoy', 'qoq', 'mom', 'eps', 'pe', 'pb',
}

def jieba_tokenizer(text):
    """中文斷詞器：過濾停用詞、單字、純數字、純標點。"""
    tokens = []
    for w in jieba.lcut(str(text).lower()):
        w = w.strip()
        if len(w) < 2 or len(w) > 10:
            continue
        if w in STOPWORDS or w in BAD_EN:
            continue
        if w.isdigit():
            continue
        if not re.search(r'[a-z\u4e00-\u9fff]', w):
            continue
        tokens.append(w)
    return tokens

# 自我驗證
# [修正 BUG-3] 分離各測試情境，確保每個過濾規則都被獨立驗證
_test = jieba_tokenizer('所以啊 ex 再來我們看 vix 2025 回答這個問題之前 去美元化')
print('jieba_tokenizer 自我驗證（混合輸入）:', _test)

# 各項過濾規則獨立驗證
_test_stopword  = jieba_tokenizer('所以這件事非常重要')   # 「所以」應被過濾
_test_bad_en    = jieba_tokenizer('ex這個問題很難回答')    # 「ex」應被過濾
_test_digit     = jieba_tokenizer('2025年的預測數字')      # 純數字「2025」應被過濾
_test_long_stop = jieba_tokenizer('回答這個問題之前先想清楚')  # 長停用詞應被過濾

assert '所以' not in _test_stopword,  'STOPWORDS 過濾失效：「所以」未被攔截'
assert 'ex'   not in _test_bad_en,   'BAD_EN 過濾失效：「ex」未被攔截'
assert '2025' not in _test_digit,    '數字過濾失效：「2025」未被攔截'
assert '回答這個問題之前' not in _test_long_stop, '長停用詞過濾失效'
print('✓ tokenizer 所有過濾規則驗證通過')


Building prefix dict from the default dictionary ...
DEBUG:jieba:Building prefix dict from the default dictionary ...
Dumping model to file cache /tmp/jieba.cache
DEBUG:jieba:Dumping model to file cache /tmp/jieba.cache
Loading model cost 0.842 seconds.
DEBUG:jieba:Loading model cost 0.842 seconds.
Prefix dict has been built successfully.
DEBUG:jieba:Prefix dict has been built successfully.


jieba_tokenizer 自我驗證（混合輸入）: ['vix', '美元']
✓ tokenizer 所有過濾規則驗證通過


## Cell 10｜BERTopic 訓練（v7：MMR + 跨主題去重）

### 關鍵字重複問題的兩層解法

**問題**：BERTopic 各主題往往共享大量高頻詞（如「市場」「美國」），導致各組難以區分。

**解法**：

1. **MaximalMarginalRelevance（MMR）**：BERTopic 內建表示層，  
   對*同一主題內*過濾語意相近的冗餘詞（`diversity=0.4` 在相關性與多樣性之間取平衡）。

2. **跨主題公共詞排除（展示層）**：  
   統計各主題前 N 詞中，哪些詞出現在 ≥ `TOP_SHARED_THRESH` 個主題，  
   在*展示*時替換為該主題排名更後面的特有詞，不修改模型本身。

兩者合用可大幅提升各主題關鍵字的區別度。


In [ ]:
# ── BERTopic 訓練（v7: 加入 MMR 去重複關鍵字）──────────────────────────────────
#
# 兩層去重複策略：
#   1. MaximalMarginalRelevance (MMR)：BERTopic 內建，
#      對「同一主題內」過濾語意高度相似的關鍵字（diversity=0.3~0.5 效果最佳）
#   2. 跨主題重複詞過濾（見下方 display_unique_keywords）：
#      統計出現在 ≥ TOP_SHARED_THRESH 個主題中的高頻詞，從各主題前 N 名中移除，
#      替換為該主題特有詞，確保各組顯示的詞彙有區別度
#
MMR_DIVERSITY    = 0.4   # 0=只看相關度；1=只看多樣性；建議 0.3~0.5
TOP_N_WORDS      = 10    # 每個主題取前 N 大詞
TOP_SHARED_THRESH = 2    # 出現在 ≥ N 個主題時視為跨主題公共詞，對顯示時排除

representation_model = MaximalMarginalRelevance(diversity=MMR_DIVERSITY)

vectorizer_zh = CountVectorizer(
    tokenizer=jieba_tokenizer,
    token_pattern=None,   # 必要：否則 sklearn 忽略自訂 tokenizer
    ngram_range=(1, 2),
)

topic_model = BERTopic(
    embedding_model=embedder,
    vectorizer_model=vectorizer_zh,
    representation_model=representation_model,   # ← MMR 去重複
    min_topic_size=10,
    # [修正 D2] 大資料集（> 2 萬筆）建議改為 False 以節省記憶體；
    #           小資料集或需要主題機率分布時可設 True。
    calculate_probabilities=True,
    verbose=True,
)

topics, probs = topic_model.fit_transform(train_docs, embeddings=train_emb)

# 用中文 vectorizer 重算一次關鍵字（確保不顯示舊填充詞）
topic_model.update_topics(
    train_docs,
    vectorizer_model=vectorizer_zh,
    representation_model=representation_model,
)

topic_info = topic_model.get_topic_info()
valid_topic_ids = [t for t in topic_model.get_topics().keys() if t != -1]
print(f'BERTopic 自動分出 {len(valid_topic_ids)} 個有效主題（-1 為離群值）')
print(topic_info[['Topic', 'Count', 'Name']].to_string(index=False))

# ── 跨主題重複詞過濾 ───────────────────────────────────────────────────────────
def display_unique_keywords(model, valid_ids, top_n=TOP_N_WORDS, shared_thresh=TOP_SHARED_THRESH):
    """
    計算各主題前 top_n 詞的跨主題出現頻率，
    把出現在 >= shared_thresh 個主題中的詞視為公共詞並從展示清單排除。
    若排除後不足 top_n，則從該主題更後面的詞補足（最多取 top_n*3 候選）。
    回傳 {topic_id: [unique_words]} 以及 shared_words set。
    """
    # 先拿足夠多的候選詞（每主題取 top_n*3 以便有替補）
    candidates = {}
    for tid in valid_ids:
        raw = model.get_topic(tid)
        candidates[tid] = [w for w, _ in raw[:top_n * 3]]

    # 統計各詞出現在幾個主題的前 top_n
    word_topic_count = Counter()
    for tid in valid_ids:
        for w in candidates[tid][:top_n]:
            word_topic_count[w] += 1

    shared_words = {w for w, cnt in word_topic_count.items() if cnt >= shared_thresh}

    # 為每個主題建立去除公共詞後的清單
    result = {}
    for tid in valid_ids:
        unique = [w for w in candidates[tid] if w not in shared_words][:top_n]
        result[tid] = unique

    return result, shared_words

unique_kw, shared_words = display_unique_keywords(topic_model, valid_topic_ids)

print(f'\n跨主題公共詞（出現在 ≥{TOP_SHARED_THRESH} 個主題，已從展示中排除）:')
print('  ' + ', '.join(sorted(shared_words)) if shared_words else '  （無公共詞）')

print(f'\n--- 各主題核心關鍵字（MMR + 跨主題去重，前 {TOP_N_WORDS} 詞）---')
_bad = {'所以', 'ex', '再來', '2025', '預期', '回答這個問題之前'}
_found_bad = False
for tid in valid_topic_ids:
    words = unique_kw[tid]
    if _bad & set(words):
        _found_bad = True
    count = topic_info.loc[topic_info['Topic'] == tid, 'Count'].values[0]
    print(f'主題 {tid:2d} (n={count:4d}): {" / ".join(words)}')

if _found_bad:
    print('\n⚠ 仍偵測到填充詞。請依序重新執行：停用詞格 → 本格。')
else:
    print('\n✓ 關鍵字已不含填充詞。')

# ── 代表性貼文抽樣 ─────────────────────────────────────────────────────────────
print('\n--- 各主題代表性貼文 ---')
for tid in valid_topic_ids:
    print(f'\n[主題 {tid}]')
    for i, doc in enumerate(topic_model.get_representative_docs(tid)[:2]):
        print(f'  代表文件 {i+1}: {doc[:300]}...')


2026-06-11 05:53:57,597 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-06-11 05:54:11,368 - BERTopic - Dimensionality - Completed ✓
2026-06-11 05:54:11,370 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-06-11 05:54:11,416 - BERTopic - Cluster - Completed ✓
2026-06-11 05:54:11,421 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-06-11 05:54:22,444 - BERTopic - Representation - Completed ✓


BERTopic 自動分出 12 個有效主題（-1 為離群值）
 Topic  Count             Name
    -1    235   -1_市場_邏輯_風險_投資
     0    145    0_市場_風險_利率_邏輯
     1     41    1_時間_人生_投資_朋友
     2     39    2_台灣_泰國_退休_越南
     3     36    3_市場_投資_風險_退休
     4     31    4_日本_台灣_匯率_市場
     5     28    5_小孩_人生_教育_老師
     6     19    6_美元_中國_川普_經濟
     7     16    7_基金_風險_市場_策略
     8     16    8_時間_市場_私募_未來
     9     13 9_油價_市場_裂解 價差_價格
    10     12   10_健康_人生_時間_每天
    11     12   11_自卑_市場_投資_人生

跨主題公共詞（出現在 ≥2 個主題，已從展示中排除）:
  人生, 價格, 利率, 台灣, 壓力, 市場, 心態, 投資, 時間, 未來, 策略, 經濟, 資產, 退休, 選擇, 邏輯, 金融, 風險

--- 各主題核心關鍵字（MMR + 跨主題去重，前 10 詞）---
主題  0 (n= 145): 估值 / 股票 / 債券
主題  1 (n=  41): 朋友 / 價值 / 妥協
主題  2 (n=  39): 泰國 / 越南 / 政府 / 今天 / 曼谷 / 人口
主題  3 (n=  36): 穩定 / 資金
主題  4 (n=  31): 日本 / 匯率 / 央行 / 泡沫 / 股市
主題  5 (n=  28): 小孩 / 教育 / 老師 / 父母 / 別人 / 家庭 / 學校 / 情感 / 學會
主題  6 (n=  19): 美元 / 中國 / 川普 / 關稅 政策 / 國債 / 債務 / 貶值
主題  7 (n=  16): 基金 / 基金 經理 / hedge / fund / 部位
主題  8 (n=  16): 私募 / 技術 分析 / neocloud / coreweave
主題  9 (n=  13): 油價 / 裂

## Cell 11｜BERTopic 套用到測試集

In [ ]:
# [修正 D] 套用 BERTopic 前先 .copy()，避免 SettingWithCopyWarning
wt_test = wt_test.copy()
test_topics, test_probs = topic_model.transform(test_docs, embeddings=test_emb)
wt_test['bertopic_topic'] = test_topics

print('測試集主題分布:')
print(wt_test['bertopic_topic'].value_counts().sort_index())


2026-06-11 05:54:40,095 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-06-11 05:54:45,281 - BERTopic - Dimensionality - Completed ✓
2026-06-11 05:54:45,282 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-06-11 05:54:45,289 - BERTopic - Probabilities - Start calculation of probabilities with HDBSCAN
2026-06-11 05:54:45,299 - BERTopic - Probabilities - Completed ✓
2026-06-11 05:54:45,300 - BERTopic - Cluster - Completed ✓


測試集主題分布:
bertopic_topic
-1    56
 0     3
 1     3
 2     4
 3     1
 4     2
 6     1
 9     2
Name: count, dtype: int64


## Cell 12｜兩種分群方法的一致性比較（ARI / NMI）

> 不把任一方法當「真實標籤」。用 ARI / NMI 觀察兩種無監督方法的一致程度。
> BERTopic 在小資料下可能大多數樣本都歸入同一主題，ARI ≈ 0 屬正常現象。


In [ ]:
# ── 過濾 BERTopic outlier（-1）再計算 ARI / NMI ────────────────────────────
# BERTopic 的 -1 代表離群文件，若直接計算會被視為獨立 cluster，人為壓低指標
_eval_mask = wt_test['bertopic_topic'] != -1
_n_outlier  = (~_eval_mask).sum()
if _n_outlier:
    print(f'⚠ BERTopic 共 {_n_outlier} 筆 outlier（topic=-1）已排除，不納入 ARI/NMI 計算')
_wt_eval = wt_test[_eval_mask]

cross_tab = pd.crosstab(
    _wt_eval['cluster'], _wt_eval['bertopic_topic'],
    rownames=['KMeans'], colnames=['BERTopic'],
)
print('KMeans 與 BERTopic 在測試集上的交叉分佈（已排除 outlier）:')
try:
    display(cross_tab)
except NameError:
    print(cross_tab.to_string())

if len(_wt_eval) < 2:
    print('⚠ 排除 outlier 後有效樣本不足，跳過 ARI/NMI 計算')
else:
    ari = adjusted_rand_score(_wt_eval['cluster'], _wt_eval['bertopic_topic'])
    nmi = normalized_mutual_info_score(_wt_eval['cluster'], _wt_eval['bertopic_topic'])
    print(f'\nAdjusted Rand Index (ARI): {ari:.4f}')
    print(f'Normalized Mutual Information (NMI): {nmi:.4f}')
    print('\n（ARI/NMI ≈ 1 代表兩方法分得很像；≈ 0 代表幾乎無關）')


⚠ BERTopic 共 56 筆 outlier（topic=-1）已排除，不納入 ARI/NMI 計算
KMeans 與 BERTopic 在測試集上的交叉分佈（已排除 outlier）:


BERTopic,0,1,2,3,4,6,9
KMeans,,,,,,,
0,0,0,0,1,0,0,0
1,0,3,0,0,0,0,0
2,1,0,0,0,0,0,0
3,2,0,0,0,2,1,2
4,0,0,4,0,0,0,0



Adjusted Rand Index (ARI): 0.4595
Normalized Mutual Information (NMI): 0.7785

（ARI/NMI ≈ 1 代表兩方法分得很像；≈ 0 代表幾乎無關）


## Cell 13｜非監督分群探索：KMeans 各群關鍵詞統計

> **v7 完全移除人工標籤流程。**  
> 本格改用純非監督方式：對 KMeans 每群的文字做 TF-IDF 式詞頻統計，  
> 列出各群最具代表性的詞彙，協助人工理解各群語意，無需任何標籤。


In [ ]:
# ── KMeans 各群關鍵詞統計（非監督：TF-IDF 群間對比）──────────────────────────
#
# 作法：對每群的文字集合，計算詞頻（TF），
# 再用「該群詞頻 / 全體平均詞頻」做群間 TF-IDF，
# 排名越高代表該詞越能區分此群與其他群。
#
TOP_K = 12   # 每群顯示前 K 個特徵詞

# 合併訓練+測試集（全資料）
all_data = pd.concat([wt_train, wt_test], ignore_index=True).copy()

# 使用全資料重新斷詞（jieba）
def tokenize_series(series):
    return [jieba_tokenizer(t) for t in series.astype(str).tolist()]

all_tokens = tokenize_series(all_data['文字內容'])

# 建立詞彙表與文件詞頻
cluster_word_freq = defaultdict(Counter)  # cluster_id → Counter(詞, 詞頻)
global_counter = Counter()

for tokens, cluster_id in zip(all_tokens, all_data['cluster']):
    cluster_word_freq[cluster_id].update(tokens)
    global_counter.update(tokens)

total_docs = len(all_data)
n_clusters_found = len(cluster_word_freq)

# 計算群內相對詞頻（群TF / 全局TF）→ 凸顯群特有詞
print(f'KMeans 各群關鍵詞（群內相對詞頻 Top {TOP_K}，已去除跨群公共詞）')
print('=' * 65)

# 先找全局高頻詞（所有群都高頻 → 區分力低，排除）
# [修正 BUG-2] 只排除「已在 STOPWORDS 中」的高頻詞，避免把「台股」「利率」等
#              有語意的主題詞誤排除。改用「出現在所有群且相對詞頻均衡」的詞作為公共詞門檻。
global_total = sum(global_counter.values())
# 新策略：全局前 50 高頻詞中，只保留確實是停用詞的詞加入排除集合
global_top = {
    w for w, c in global_counter.most_common(50)
    if w in STOPWORDS  # 只排除已知停用詞，保留有語意的高頻詞
}

cluster_top_words = {}
for cid in sorted(cluster_word_freq.keys()):
    ctr = cluster_word_freq[cid]
    c_total = sum(ctr.values())
    # 相對詞頻 = 群內詞頻比例 / 全局詞頻比例
    scores = {}
    for w, cnt in ctr.items():
        if w in global_top:
            continue
        tf_local  = cnt / c_total if c_total else 0
        tf_global = global_counter[w] / global_total if global_total else 1e-9
        scores[w] = tf_local / (tf_global + 1e-9)
    top_words = sorted(scores, key=scores.get, reverse=True)[:TOP_K]
    cluster_top_words[cid] = top_words
    cluster_size = (all_data['cluster'] == cid).sum()
    print(f'\n群 {cid} (n={cluster_size}):')
    print('  ' + ' / '.join(top_words))

# ── 非監督分群穩定性：Bootstrap ARI ─────────────────────────────────────────
print('\n' + '=' * 65)
print('Bootstrap 分群穩定性（重複抽樣 KMeans，計算 ARI 分布）')
print('ARI 越接近 1 表示兩次分群高度一致（分群穩定）')

N_BOOTSTRAP = 20
ari_list = []
n_train = len(train_emb)
rng = np.random.default_rng(RANDOM_STATE)

ref_labels = km.labels_.copy()
for _ in range(N_BOOTSTRAP):
    idx = rng.choice(n_train, size=n_train, replace=True)
    km_b = KMeans(n_clusters=N_CLUSTERS, n_init=10, random_state=int(rng.integers(9999))).fit(train_emb[idx])  # [修正 D1] n_init 對齊訓練設定
    # 只取 bootstrap 樣本中重複出現的 idx 做評估（取 unique）
    unique_idx = np.unique(idx)
    pred_b = km_b.predict(train_emb[unique_idx])
    ari_list.append(adjusted_rand_score(ref_labels[unique_idx], pred_b))

ari_arr = np.array(ari_list)
print(f'  ARI 均值: {ari_arr.mean():.4f}  標準差: {ari_arr.std():.4f}')
print(f'  ARI 範圍: [{ari_arr.min():.4f}, {ari_arr.max():.4f}]')
if ari_arr.mean() > 0.6:
    print('  ✓ 分群結構穩定')
elif ari_arr.mean() > 0.3:
    print('  ⚠ 分群結構中等穩定，建議嘗試不同 k 值')
else:
    print('  ✗ 分群不穩定，建議調整 k 或 embedding 模型')


KMeans 各群關鍵詞（群內相對詞頻 Top 12，已去除跨群公共詞）

群 0 (n=134):
  自卑 / cadiz / 30k / 月入 / monash / af / 球場 / 大姐 / 囤幣 / overpriced / 合夥 / 圈養

群 1 (n=165):
  meta / neocloud / coreweave / 古龍 / 落魄 / 象限 / blue / 問者 / 老三 / 寫作 / 抱團 / owl

群 2 (n=190):
  epv / 港股 / av / greenwald / qr / waymo / 裴洛西 / 英特 / franchise / avis / coinbase / 股價會

群 3 (n=148):
  汽油 / 柴油 / 存底 / 岩油 / 逆差 / rrp / 美國頁 / 航煤 / 煉廠 / duc / 升貼 / 經常帳

群 4 (n=78):
  國民黨 / 子公司 / 統治 / 焦土政策 / 生育率 / 賣國 / 家父 / 觀光 / 高技能 / 不生 / cybertruck / 三丰

Bootstrap 分群穩定性（重複抽樣 KMeans，計算 ARI 分布）
ARI 越接近 1 表示兩次分群高度一致（分群穩定）
  ARI 均值: 0.4521  標準差: 0.1047
  ARI 範圍: [0.2823, 0.7010]
  ⚠ 分群結構中等穩定，建議嘗試不同 k 值


## Cell 14｜主題視覺化

In [ ]:
_valid_topics = topic_info[topic_info['Topic'] != -1]

# 主題分佈圖（需 >= 2 個有效主題）
if len(_valid_topics) >= 2:
    fig_topics = topic_model.visualize_topics()
    fig_topics.show()
else:
    print(f'⚠ BERTopic 僅找到 {len(_valid_topics)} 個有效主題，'
          '無法繪製主題分佈圖（需 >= 2 個主題）。')

# 各主題關鍵字長條圖
fig_bar = topic_model.visualize_barchart(top_n_topics=8)
fig_bar.show()


## 結論（v7.2 修正內容）

**v7.2 相較 v7.1 的修正項目：**  
- BUG-1：Cell 1 加入 `scikit-learn>=1.0` 安裝，確保 `token_pattern=None` 可用  
- BUG-2：Cell 13 `global_top` 改為只排除 STOPWORDS 中的詞，避免誤砍有語意的高頻詞  
- BUG-3：Cell 9 assert 測試分離各情境，真正驗證每條過濾規則  
- D1：Bootstrap ARI 的 `n_init` 從 5 對齊至 10  
- D2：`calculate_probabilities` 加入大資料集記憶體警告  
- D3：`BAD_EN` 擴充財經常見英文縮寫（tf, gdp, ipo, etf 等）

## 結論

| 項目 | v6.2 | v7.0（本版）|
|---|---|---|
| **評估架構** | 人工標籤 + Stratified K-Fold | **純非監督**：Silhouette / CH / DB 三指標 + Bootstrap ARI（n_init=10）|
| **分群理解** | 人工類別映射 | **KMeans 群內相對 TF-IDF** 自動提取各群特徵詞 |
| **關鍵字重複** | 無處理，各主題詞彙大量交疊 | **MMR（主題內去重）+ 跨主題公共詞排除（展示層）** |
| BERTopic 訓練 | 無 diversity 設定 | `MaximalMarginalRelevance(diversity=0.4)` |
| 分群品質評估 | 僅 Silhouette | **Silhouette + Calinski-Harabasz + Davies-Bouldin** |
| 分群穩定性 | 無 | **Bootstrap ARI（n=20）** |
| `StratifiedKFold` | ✓（人工標籤） | 移除（無監督不需要） |
| `classification_report` | ✓（人工標籤） | 移除 |
| sklearn 相容性 | `n_init=10` ✓ | 維持（`scikit-learn>=1.0` 明確指定）|
| 停用詞 / jieba | ✓ | BAD_EN 擴充財經縮寫；assert 測試強化 |
| 多語言 Embedding | ✓ | 維持 |

**如何解讀分群結果（無監督版本）**  
1. **Cell 8 三指標**：選擇使 Silhouette 最大、DB 最小的 k 值。  
2. **Cell 10 BERTopic 關鍵字**：觀察各主題前10大詞，人工命名語意。  
3. **Cell 13 群內 TF-IDF**：以相對詞頻確認 KMeans 各群的主要語意方向。  
4. **Bootstrap ARI**：確認分群結構是否穩健，ARI > 0.6 為可接受門檻。  
